# Pipeline 01: Limpieza, Modelado y Generación de Datos

Este notebook se encarga de todo el flujo de ingesta, procesamiento y modelado de datos.

### Fuentes de Datos:
Se ingieren los siguientes archivos desde data/raw/:
* clientes.csv: Datos demográficos y socioeconómicos.
* creditos.csv: Historial y originación de créditos.
* pagos.csv: Transacciones y estado de cuotas.
* ventos_app.csv: Comportamiento digital del cliente en la aplicación.

### Resultados del pipeline:

| Artefacto Generado | Ubicación | Descripción |
| :--- | :--- | :--- |
| **Tablas Raw** | Supabase (
aw_*) | Tablas crudas originales con tipos inferidos |
| **Vista BI** | Supabase (_clientes_bi) | Vista saneada para Power BI con manejo de outliers (ingreso_mensual) |
| **Predicciones DB** | Supabase (predicciones_riesgo) | Tabla final analítica con cálculo de variable objetivo y predicción LLM |
| **Mini Data Lake** | data/processed/powerbi/*.csv | Tablas dimensionales en CSV configuradas para BI sin ODBC |
| **Modelo Entrenado** | models/modelo_mora.pkl | LightGBM empaquetado para despliegue |


In [23]:
import sys
import os
import importlib
from pathlib import Path
from dotenv import load_dotenv

current_dir = Path.cwd()
ROOT_DIR = current_dir if (current_dir / "src").exists() else current_dir.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

load_dotenv(ROOT_DIR / ".env", override=True)

import src.config as _cfg_module
importlib.reload(_cfg_module)
from src.config import settings
from src.db_utils import create_supabase_engine, get_database_host, table_counts

engine = create_supabase_engine(settings.DATABASE_URL)
print(f"Conectado a Supabase/PostgreSQL: {get_database_host(settings.DATABASE_URL)}")
print(f"Directorio raw: {settings.DATA_RAW_DIR}")

Conectado a Supabase/PostgreSQL: aws-1-us-east-1.pooler.supabase.com
Directorio raw: C:\Daniel\Mio\project-data-scientist\data\raw


## 1. Cargar datos crudos en Supabase

Carga idempotente desde `data/raw/` hacia `raw_clientes`, `raw_creditos`, `raw_pagos` y `raw_eventos_app`.

In [24]:
from scripts.load_raw_supabase import main as run_carga_raw

run_carga_raw()

raw_tables = ["raw_clientes", "raw_creditos", "raw_pagos", "raw_eventos_app"]
print("\nConteo posterior a carga raw:")
print(table_counts(engine, raw_tables))

2026-05-21 20:02:54,386 - src.ingesta - INFO - Cargando todos los archivos del dataset...
2026-05-21 20:02:54,386 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\clientes.csv
2026-05-21 20:02:54,399 - src.ingesta - INFO - Columna 'fecha_registro' casteada a datetime64[ns].
2026-05-21 20:02:54,400 - src.ingesta - INFO - Archivo clientes.csv cargado exitosamente. Forma: (1400, 16)
2026-05-21 20:02:54,401 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\creditos.csv
2026-05-21 20:02:54,413 - src.ingesta - INFO - Columna 'fecha_desembolso' casteada a datetime64[ns].
2026-05-21 20:02:54,414 - src.ingesta - INFO - Archivo creditos.csv cargado exitosamente. Forma: (1527, 13)
2026-05-21 20:02:54,415 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\pagos.csv
2026-05-21 20:02:54,449 - src.ingesta - INFO - Columna 'fecha_vencimiento' casteada a datetime6

Base de datos activa: aws-1-us-east-1.pooler.supabase.com
raw_clientes: 1400 filas fuente, 1400 filas insertadas/actualizadas
raw_creditos: 1527 filas fuente, 1527 filas insertadas/actualizadas
raw_pagos: 10434 filas fuente, 10434 filas insertadas/actualizadas
raw_eventos_app: 13324 filas fuente, 13324 filas insertadas/actualizadas
          tabla  filas_en_supabase
   raw_clientes               1400
   raw_creditos               1527
      raw_pagos              10434
raw_eventos_app              13324

Conteo posterior a carga raw:
{'raw_clientes': np.int64(1400), 'raw_creditos': np.int64(1527), 'raw_pagos': np.int64(10434), 'raw_eventos_app': np.int64(13324)}


## 2. Crear vista analítica para Power BI

La prueba requiere una capa de visualización. Esta vista prepara el campo de ingreso para BI sin modificar los datos raw: conserva el dato original, capea el valor de presentación al P99 y deja una bandera auditable de outlier.

In [25]:
import pandas as pd
from sqlalchemy import text

sql_view = """
CREATE OR REPLACE VIEW v_clientes_bi AS
WITH p AS (
    SELECT percentile_cont(0.99) WITHIN GROUP (ORDER BY ingreso_mensual_estimado) AS p99
    FROM raw_clientes
    WHERE ingreso_mensual_estimado IS NOT NULL
)
SELECT
    c.*,
    LEAST(c.ingreso_mensual_estimado, (SELECT p99 FROM p)) AS ingreso_mensual_bi,
    CASE
        WHEN c.ingreso_mensual_estimado > (SELECT p99 FROM p) AND COALESCE(c.estrato, 4) <= 3
            THEN 'error_captura'
        WHEN c.ingreso_mensual_estimado > (SELECT p99 FROM p)
            THEN 'outlier_real'
        ELSE 'normal'
    END AS flag_ingreso
FROM raw_clientes c;
"""

with engine.begin() as conn:
    conn.execute(text(sql_view))

print("Vista v_clientes_bi creada/actualizada.")

df_outliers = pd.read_sql("""
SELECT cliente_id, ingreso_mensual_estimado, ingreso_mensual_bi, flag_ingreso, estrato, ocupacion
FROM v_clientes_bi
WHERE flag_ingreso IN ('error_captura', 'outlier_real')
ORDER BY ingreso_mensual_estimado DESC;
""", engine)
display(df_outliers)

Vista v_clientes_bi creada/actualizada.


,cliente_id,ingreso_mensual_estimado,ingreso_mensual_bi,flag_ingreso,estrato,ocupacion
0,CL00493,120000000.0,7152200.0,error_captura,2,Independiente
1,CL00353,10000000.0,7152200.0,outlier_real,4,Empleado
2,CL00686,9450000.0,7152200.0,outlier_real,4,Empleado
3,CL01100,9200000.0,7152200.0,outlier_real,4,Empleado
4,CL00459,9110000.0,7152200.0,outlier_real,5,Empleado
5,CL00258,9000000.0,7152200.0,error_captura,3,Microempresario
6,CL00504,8510000.0,7152200.0,error_captura,2,Empleado
7,CL00676,8260000.0,7152200.0,outlier_real,4,Empleado
8,CL01182,8240000.0,7152200.0,outlier_real,5,Empleado
9,CL01197,7790000.0,7152200.0,outlier_real,5,Empleado


## 3. Entrenar modelo, cargar predicciones y exportar mini data lake

Construye la ABT, entrena LightGBM, serializa `models/modelo_mora.pkl`, hace upsert de `predicciones_riesgo` y exporta tablas CSV en `data/processed/powerbi/` para que el `.pbix` funcione sin ODBC.

In [30]:
import importlib
import scripts.load_powerbi_predictions as _powerbi_predictions

importlib.reload(_powerbi_predictions)
run_predicciones = _powerbi_predictions.main

run_predicciones()

print("\nConteo posterior a predicciones:")
print(table_counts(engine, ["predicciones_riesgo"]))

powerbi_dir = settings.DATA_PROCESSED_DIR / "powerbi"
print(f"\nMini data lake local para Power BI: {powerbi_dir}")
for path in sorted(powerbi_dir.glob("*.CSV")):
    print(f"- {path.name}")

2026-05-21 20:13:22,858 - src.ingesta - INFO - Cargando todos los archivos del dataset...
2026-05-21 20:13:22,860 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\clientes.csv
2026-05-21 20:13:22,870 - src.ingesta - INFO - Columna 'fecha_registro' casteada a datetime64[ns].
2026-05-21 20:13:22,871 - src.ingesta - INFO - Archivo clientes.csv cargado exitosamente. Forma: (1400, 16)
2026-05-21 20:13:22,871 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\creditos.csv
2026-05-21 20:13:22,878 - src.ingesta - INFO - Columna 'fecha_desembolso' casteada a datetime64[ns].
2026-05-21 20:13:22,879 - src.ingesta - INFO - Archivo creditos.csv cargado exitosamente. Forma: (1527, 13)
2026-05-21 20:13:22,880 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\pagos.csv
2026-05-21 20:13:22,910 - src.ingesta - INFO - Columna 'fecha_vencimiento' casteada a datetime6

Base de datos activa: aws-1-us-east-1.pooler.supabase.com


2026-05-21 20:13:23,051 - src.train_mora - INFO - Dataset particionado. Train=1221, Test=306.
2026-05-21 20:13:23,052 - src.train_mora - INFO - Features utilizadas en el modelo (35): ['producto_credito', 'monto_credito', 'plazo_meses', 'tasa_interes_mensual', 'valor_cuota_pactada', 'canal_originacion', 'score_interno_originacion', 'relacion_cuota_ingreso', 'politica_aprobacion', 'departamento', 'ciudad', 'edad', 'genero', 'estrato', 'nivel_educativo', 'ocupacion', 'ingreso_mensual_estimado', 'canal_adquisicion', 'score_externo', 'tiene_producto_ahorro', 'numero_dependientes', 'dispositivo_principal', 'antiguedad_cliente_dias', 'mes_desembolso', 'dia_semana_desembolso', 'prev_evento_actualizacion_datos', 'prev_evento_consulta_saldo', 'prev_evento_login', 'prev_evento_pago_exitoso', 'prev_evento_pago_fallido', 'prev_evento_pago_iniciado', 'prev_evento_simulacion_credito', 'prev_evento_solicitud_soporte', 'prev_evento_sesion_seg_tot', 'prev_evento_sesion_seg_avg']
2026-05-21 20:13:23,053 

[LightGBM] [Info] Number of positive: 333, number of negative: 888
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000682 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2556
[LightGBM] [Info] Number of data points in the train set: 1221, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best